# 04. FAISSインデックス構築

staging データから FAISS インデックスを構築して `data/staging/faiss/` に保存する。
**本番インデックスへの書き込みは行わない。** 本番反映は `05_export_to_fastapi.ipynb` で実施。

In [ ]:
DRY_RUN = False

import sys, json, logging
from pathlib import Path

try:
    BASE
except NameError:
    BASE        = Path("/content/AI_TradeManagement")
    STAGING_DIR = BASE / "data" / "staging"
    sys.path.insert(0, str(BASE / "scripts"))

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
FAISS_OUT = STAGING_DIR / "faiss"
FAISS_OUT.mkdir(parents=True, exist_ok=True)
print(f"DRY_RUN={DRY_RUN}, FAISS_OUT={FAISS_OUT}")

## 1. 制裁エンティティ FAISS 構築

In [ ]:
from pipeline.index.faiss_builder import build_sanctions_index

merged_path = STAGING_DIR / "sanctions" / "sanctions_merged.json"
if not merged_path.exists():
    print("⚠️  sanctions_merged.json なし — 02 を先に実行してください")
else:
    entities = json.loads(merged_path.read_text())
    print(f"  入力: {len(entities):,} エンティティ")
    result = build_sanctions_index(entities=entities, output_dir=FAISS_OUT, dry_run=DRY_RUN)
    if result:
        ntotal, idx_path, _ = result
        print(f"✅ entities.index 構築完了: ntotal={ntotal:,} ({idx_path.stat().st_size:,} bytes)")

## 2. 規制マトリクス FAISS 構築

In [ ]:
import sqlite3

DB_PATH = BASE / "modules" / "ai_validation" / "app.db"
if not DB_PATH.exists():
    print(f"⚠️  DB not found: {DB_PATH}")
else:
    conn = sqlite3.connect(str(DB_PATH))
    rows = conn.execute(
        "SELECT id, title, requirement_text, usage_criteria_text, "
        "tech_criteria_text, notes, item_no, list_name FROM matrix_rules"
    ).fetchall()
    conn.close()
    cols = ["rule_id","title","requirement_text","usage_criteria_text",
            "tech_criteria_text","notes","item_no","list_name"]
    matrix_rules = [dict(zip(cols, r)) for r in rows]
    print(f"  入力: {len(matrix_rules)} ルール")

    from pipeline.index.faiss_builder import build_matrix_rules_index
    result = build_matrix_rules_index(matrix_rules=matrix_rules, output_dir=FAISS_OUT, dry_run=DRY_RUN)
    if result:
        ntotal, idx_path, _ = result
        print(f"✅ matrix_rules.index 構築完了: ntotal={ntotal}")

## 3. サマリー

In [ ]:
for f in sorted(FAISS_OUT.rglob("*")):
    if f.is_file():
        print(f"  {f.name}  ({f.stat().st_size:,} bytes)")
print("\n次のノートブック → 05_export_to_fastapi.ipynb")